# **Merge, Join, and Concatenate**

**DS112 — Computer Programming**

**Week 15:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Demonstrate merging tools on toy and real-life datasets.

Real data is often spread across **several tables**. The canteen's sales file records item **codes** and quantities, but the **prices** and **names** are in a separate menu file. To compute revenue we must **combine** them.

| Operation | What it does | Analogy |
| --------- | ------------ | ------- |
| `pd.concat` | **stacks** tables on top of (or beside) each other | stapling two attendance sheets together |
| `pd.merge` | **matches rows** using a common key | looking up each student ID in the registrar's list |
| `df.join` | merge using the **index** | the same, keyed by row labels |

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

sales = pd.read_csv("data/canteen_sales.csv")
menu = pd.read_csv("data/menu.csv")
print(sales.head(3))
print(menu.head(3))

## **Concatenation**

In [ ]:
ds1a = pd.DataFrame({"name": ["Ana", "Ben"], "final": [85, 72]})
ds1b = pd.DataFrame({"name": ["Carlo", "Diana"], "final": [94, 66]})

stacked = pd.concat([ds1a, ds1b], keys=["DS1A", "DS1B"], names=["section", None])
stacked

In [ ]:
quiz = pd.DataFrame({"quiz": [88, 75]}, index=["Ana", "Ben"])
lab = pd.DataFrame({"lab": [92, 80]}, index=["Ana", "Ben"])
pd.concat([quiz, lab], axis=1)                 # side by side, aligned on the index

## **Merging on a Key**

`pd.merge(left, right, on="key", how=…)` matches rows whose keys are equal. The `how` argument decides what happens to rows **without** a match.

```text
 left (students)          right (enrollment)
 id    name               id    course
 1     Ana                1     DS112
 2     Ben                1     MATH101
 3     Carlo              3     DS112
                          4     PE1

 inner → ids 1, 3        (only matches)
 left  → ids 1, 2, 3     (all of left; Ben's course is NaN)
 right → ids 1, 3, 4     (all of right; id 4's name is NaN)
 outer → ids 1, 2, 3, 4  (everything)
```

In [ ]:
students = pd.DataFrame({"id": [1, 2, 3], "name": ["Ana", "Ben", "Carlo"]})
enrollment = pd.DataFrame({"id": [1, 1, 3, 4], "course": ["DS112", "MATH101", "DS112", "PE1"]})

for how in ["inner", "left", "right", "outer"]:
    print(f"--- {how} ---")
    print(pd.merge(students, enrollment, on="id", how=how))

```{mermaid}
flowchart LR
    I["inner<br/>only matching keys"] --- L["left<br/>all left rows"]
    L --- R["right<br/>all right rows"]
    R --- O["outer<br/>all rows from both"]
```

### **Checking a merge**

* `indicator=True` adds a `_merge` column showing where each row came from.
* `validate="many_to_one"` raises an error if the right table unexpectedly has duplicate keys.

In [ ]:
pd.merge(students, enrollment, on="id", how="outer", indicator=True)

## **Real Example: Canteen Revenue**

In [ ]:
detailed = sales.merge(menu, on="item_code", how="left", validate="many_to_one")
detailed["revenue"] = detailed["qty"] * detailed["price"]
print(detailed["item"].isna().sum(), "sales without a menu match")
detailed.head()

In [ ]:
(
    detailed.groupby(["category", "item"])
    .agg(units=("qty", "sum"), revenue=("revenue", "sum"))
    .sort_values("revenue", ascending=False)
)

### **Different key names and suffixes**

In [ ]:
targets = pd.DataFrame({"code": ["RICE", "ADOBO", "HALO"], "monthly_target": [2500, 1200, 500]})
units = detailed.groupby("item_code", as_index=False)["qty"].sum()

check = units.merge(targets, left_on="item_code", right_on="code", how="inner")
check["met_target"] = check["qty"] >= check["monthly_target"]
check

When both tables have a non-key column with the same name, pandas adds `_x` and `_y` (change them with `suffixes=("_left", "_right")`).

### **Finding rows with no match (anti-join)**

In [ ]:
sold_codes = set(sales["item_code"])
menu[~menu["item_code"].isin(sold_codes)]          # menu items never sold (none in this dataset)

## **Joining on the Index**

In [ ]:
prices = menu.set_index("item_code")["price"]
units_by_code = sales.groupby("item_code")["qty"].sum().to_frame("units")
units_by_code.join(prices).assign(revenue=lambda d: d["units"] * d["price"])

## **Common Pitfalls**

* **Duplicate keys multiply rows.** If a key appears twice on each side, the result has four rows. Use `validate=` to catch it.
* **Mismatched types.** `"001"` (text) does not match `1` (number). Check `dtypes` before merging.
* **Spaces and case.** `"Rice "` ≠ `"rice"`. Clean keys with `.str.strip().str.upper()`.
* **Silent drops.** An `inner` merge quietly removes unmatched rows; compare row counts before and after.

## **Summary**

| Task | Code |
| ---- | ---- |
| stack rows / columns | `pd.concat([a, b])`, `pd.concat([a, b], axis=1)` |
| match on a column | `pd.merge(a, b, on="key", how="inner/left/right/outer")` |
| different key names | `left_on=`, `right_on=` |
| match on the index | `a.join(b)` |
| check a merge | `indicator=True`, `validate="many_to_one"` |

## **References**

Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM, 13*(6), 377–387. https://doi.org/10.1145/362384.362685

Maulion, M. R. (2022). *DS 112 computer programming* [Instructional material]. University of Science and Technology of Southern Philippines.

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/

The pandas development team. (2026). *Merge, join, concatenate and compare*. https://pandas.pydata.org/docs/user_guide/merging.html

VanderPlas, J. (2023). *Python data science handbook: Essential tools for working with data* (2nd ed.). O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/